In [1]:
%pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-ollama pypdf

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.5.2-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached sqlalchemy-2.0.52-cp313-cp313-win_amd64.whl.metadata (9.9 kB)
  Using cached greenlet-3.5.5-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 12.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 11.9 MB/s  0:00:00
Using cached sqlalchemy-2.0.52-cp313-cp313-win_amd64.whl (2.2 MB)
Using cached greenlet-3.5.5-cp313-cp313-win_amd64.whl (324 kB)
Using cached numpy-2.5.2-cp313-cp313-win_amd64.whl (12.5 MB)

   ---------------------------------------- 0/9 [pypdf]
   ---------------------------------------- 0/9 [pypdf]
   ---------------------------------------

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader

def load_documents(folder_path: str):
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder '{folder_path}' does not exist")

    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            print(f"📄 Loading: {filename}")
            try:
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            except Exception as e:
                print(f"❌ Error loading {filename}: {e}")
    return documents

C:\Users\pulki\AppData\Local\Temp\ipykernel_27228\2876647061.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\pulki\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_text(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    chunks = splitter.split_documents(documents)
    print(f"✂️ Created {len(chunks)} chunks")
    return chunks

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9215.10it/s]


In [5]:
from langchain_chroma import Chroma

def create_vector_store(chunks):
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory="./chroma_db",
        collection_name="rag_docs"
    )
    return vector_store

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def query_rag_system(query_text, vector_store):
    llm = ChatOllama(model="llama3") # Make sure you have Ollama installed and running!

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    prompt = ChatPromptTemplate.from_template(
        """
        You are a helpful assistant.
        Answer ONLY using the context below.
        If the answer is not present, say "I don't know."

        Context:
        {context}

        Question:
        {question}
        """
    )

    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain.invoke(query_text)

In [11]:
import os
from langchain_chroma import Chroma

def main():
    folder_path = "./data"

    if not os.path.exists("./chroma_db"):
        print("📦 No vector DB found. Creating one...")

        docs = load_documents(folder_path)
        chunks = split_text(docs)
        vector_store = create_vector_store(chunks)

        print("Vector database created")

    else:
        print("📦 Loading existing vector DB...")

        vector_store = Chroma(
            persist_directory="./chroma_db",
            embedding_function=embedding_function,
            collection_name="rag_docs"
        )

    while True:
        query = input("\n❓ Ask a question (or type 'exit'): ")

        if query.lower().strip() == "exit":
            break

        print("🤔 Thinking...")

        answer = query_rag_system(query, vector_store)

        print("\n🧠 Answer:\n", answer)


if __name__ == "__main__":
    main()

📦 No vector DB found. Creating one...
✂️ Created 0 chunks


ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.